# E3 — Version cible exécutable

Copie dérivée de `E3-model_1.ipynb`, avec la structure des étapes d’origine conservée. Les cellules remplacées portent des commentaires **CONFLIT** : SUPPRESSION, MODIFICATION ou AJOUT.

**Exécution :** lancer les cellules dans l’ordre, sur un kernel vierge. Vérifier `DATA_DIR` et les chemins `filepath` avant le `fit()`. Le notebook refuse de démarrer si les effectifs du split commun ne correspondent pas. Le test est chargé pour contrôler le split, **jamais évalué**. Les modèles sont réentraînés ; aucun résultat ancien n’est réutilisé.

**CONFLIT — AJOUT :** paramétrer `DATA_DIR` (CSV) et `DATASET_ROOT` (dossier des 4 classes) dans la première cellule. Chemins CSV Windows relatifs convertis sans changer les CSV. Affectations train, validation et test séparées.


In [ ]:
# CONFLIT — MODIFICATION : chargement commun et contrôle des splits avant entraînement.
# CONFLIT — MODIFICATION : changer uniquement PROJECT_DIR pour CSV et images.
from pathlib import Path
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# CONFLIT — MODIFICATION : un seul dossier Drive imposé pour les CSV et images.
# Monter Google Drive dans Colab AVANT cette cellule :
# from google.colab import drive
# drive.mount("/content/drive")
# Modifier uniquement PROJECT_DIR selon le dossier partagé.
PROJECT_DIR = Path("/content/drive/MyDrive/projet_covid")
DATA_DIR = PROJECT_DIR
DATASET_ROOT = PROJECT_DIR / "COVID-19_Radiography_Dataset"
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Dossier projet introuvable : {DATA_DIR}. Monter Drive et vérifier PROJECT_DIR.")
if not DATASET_ROOT.is_dir():
    raise FileNotFoundError(f"Dossier images introuvable : {DATASET_ROOT}. Respecter l'arborescence Kaggle.")
OUTPUT_DIR = Path(".")  # CONFLIT — MODIFICATION : sorties locales, sans chemin Drive fragile.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df = pd.read_csv(DATA_DIR / "validation.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

# CONFLIT — AJOUT : ces effectifs sont ceux du split de référence E0/E4.
EXPECTED = {"train": 13545, "validation": 3387, "test": 4233}
for name, frame in (("train", train_df), ("validation", val_df), ("test", test_df)):
    assert {"filepath", "target"}.issubset(frame.columns), f"{name}: colonnes filepath/target manquantes"
    assert len(frame) == EXPECTED[name], f"CONFLIT split {name}: {len(frame)} au lieu de {EXPECTED[name]}"
    assert frame["filepath"].notna().all() and frame["target"].notna().all(), f"{name}: valeurs manquantes"
    assert set(frame["target"].astype(str).unique()) <= {"0", "1"}, f"{name}: target non binaire"
    assert frame["filepath"].is_unique, f"{name}: chemins dupliqués"
    print(f"{name}: {len(frame)} images; COVID={sum(frame['target'].astype(str)=='1')}")
train_paths, val_paths, test_paths = (set(d["filepath"].astype(str)) for d in (train_df, val_df, test_df))
assert not (train_paths & val_paths or train_paths & test_paths or val_paths & test_paths), "CONFLIT : images présentes dans plusieurs splits"
# CONFLIT — AJOUT : mêmes effectifs et absence de chevauchement ne prouvent pas
# l'identité image par image avec les CSV d'un autre membre de l'équipe.

# CONFLIT — MODIFICATION : affectations explicites, séparées et cohérentes avec E0.
train_gen_df = train_df.copy()
val_gen_df = val_df.copy()
test_gen_df = test_df.copy()

# CONFLIT — AJOUT : conversion des étiquettes pour les générateurs binaires.
train_gen_df["target_str"] = train_gen_df["target"].astype(str)
val_gen_df["target_str"] = val_gen_df["target"].astype(str)
test_gen_df["target_str"] = test_gen_df["target"].astype(str)

# CONFLIT — AJOUT : les CSV contiennent des chemins relatifs Windows comme
# ..\covid\COVID-19_Radiography_Dataset\Lung_Opacity\images\Lung_Opacity-303.png
# Sur Colab / Drive, définir la racine du dataset (dossier contenant les 4 classes).
# Utiliser uniquement PROJECT_DIR, défini ci-dessus.
# CONFLIT — SUPPRESSION : DATASET_ROOT défini uniquement à partir de PROJECT_DIR ci-dessus.

# CONFLIT — MODIFICATION : reconstituer les chemins depuis le dossier dataset,
# sans toucher aux CSV ni modifier les appartenances train/validation/test.
def resolve_image_path(csv_path):
    normalized = str(csv_path).replace("\\", "/")
    marker = "COVID-19_Radiography_Dataset/"
    if marker not in normalized:
        raise ValueError(f"CONFLIT : chemin CSV sans dossier dataset : {csv_path}")
    relative = normalized.split(marker, 1)[1]
    parts = Path(relative).parts
    if len(parts) != 3 or parts[1] != "images" or parts[0] not in {
        "COVID", "Normal", "Lung_Opacity", "Viral Pneumonia"
    }:
        raise ValueError(f"CONFLIT : structure d'image inattendue : {csv_path}")
    return str(DATASET_ROOT.joinpath(*parts))

train_gen_df["filepath"] = train_gen_df["filepath"].map(resolve_image_path)
val_gen_df["filepath"] = val_gen_df["filepath"].map(resolve_image_path)
test_gen_df["filepath"] = test_gen_df["filepath"].map(resolve_image_path)

# CONFLIT — AJOUT : vérifier les trois ensembles, y compris le test non utilisé à l'entraînement.
print("DATASET_ROOT :", DATASET_ROOT)
for name, frame in (("train", train_gen_df), ("validation", val_gen_df), ("test", test_gen_df)):
    missing = [p for p in frame["filepath"] if not Path(p).is_file()]
    if missing:
        raise FileNotFoundError(
            f"CONFLIT {name}: {len(missing)} images introuvables ; exemple : {missing[0]}. "
            "Vérifier PROJECT_DIR et la structure Kaggle avant de lancer fit()."
        )
    print(f"{name} : {len(frame)} chemins vérifiés")

BATCH_SIZE = 32
IMG_HEIGHT, IMG_WIDTH = 224, 224
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(42)


In [ ]:
# CONFLIT — SUPPRESSION : première expérience OS seul (E1), hors périmètre E3.
# Les étapes et le CNN sont conservés ci-dessous pour E3 uniquement.

In [ ]:
# CONFLIT — SUPPRESSION : anciens générateurs OS seul (E1).

In [ ]:
# Étape 2 : Architecture du CNN E3
# CONFLIT — MODIFICATION : même architecture que E0.

In [ ]:
# CONFLIT — MODIFICATION : architecture STRICTEMENT identique à E0.
# CONFLIT — SUPPRESSION : BatchNormalization supprimée, car absente de E0.
# CONFLIT — MODIFICATION : Adam 1e-3 au lieu de 1e-4; mêmes métriques que E0.
def build_cnn_model():
    model = models.Sequential([
        layers.Input(shape=(224, 224, 1)),
        layers.Conv2D(32, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(128, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(1, activation="sigmoid")
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.Precision(name="precision"),
                 tf.keras.metrics.Recall(name="recall"), tf.keras.metrics.AUC(name="auc")]
    )
    return model

model = build_cnn_model()
model.summary()


In [ ]:
# CONFLIT — SUPPRESSION : entraînement OS seul (E1), hors périmètre E3.

In [ ]:
# CONFLIT — SUPPRESSION : fit OS seul (E1) ; évite un entraînement supplémentaire.

In [ ]:
# CONFLIT — SUPPRESSION : courbes OS seul (E1).

In [ ]:
# CONFLIT — SUPPRESSION : évaluation OS seul (E1).

In [ ]:
# CONFLIT — SUPPRESSION : métriques OS seul (E1).

In [ ]:
# CONFLIT — SUPPRESSION : ancienne heatmap OS seul (E1).

In [ ]:
# CONFLIT — SUPPRESSION : ancienne matrice de confusion OS seul (E1).

In [ ]:
# Étape 6 : Générateurs E3 avec OS + DA
# CONFLIT — MODIFICATION : entraînement E3 uniquement.

In [ ]:
# CONFLIT — AJOUT : oversampling aléatoire UNIQUEMENT sur le train, sans fuite de validation/test.
# CONFLIT — MODIFICATION : le train suréchantillonné est construit explicitement.
from sklearn.utils import resample
train_gen_df = train_gen_df.copy()
class_counts = train_gen_df["target_str"].value_counts()
majority_n = int(class_counts.max())
parts = []
for label in ("0", "1"):
    subset = train_gen_df[train_gen_df["target_str"] == label]
    assert len(subset) > 0, f"CONFLIT : classe {label} absente du train"
    parts.append(resample(subset, replace=True, n_samples=majority_n, random_state=42))
train_df_resampled = pd.concat(parts, ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
print("Train avant OS:", train_gen_df["target_str"].value_counts().to_dict())
print("Train après OS:", train_df_resampled["target_str"].value_counts().to_dict())
assert len(train_df_resampled) == 2 * majority_n

# CONFLIT — MODIFICATION : augmentation appliquée uniquement au train.
# CONFLIT — SUPPRESSION : horizontal_flip=True remplacé par False (radiographies).
train_datagen = ImageDataGenerator(
    rescale=1./255, rotation_range=10,
    width_shift_range=0.1, height_shift_range=0.1,
    zoom_range=0.1, horizontal_flip=False
)
val_datagen = ImageDataGenerator(rescale=1./255)
train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df_resampled, x_col="filepath", y_col="target_str",
    target_size=(IMG_HEIGHT, IMG_WIDTH), color_mode="grayscale",
    class_mode="binary", classes=["0", "1"], batch_size=BATCH_SIZE,
    shuffle=True, seed=42
)
val_generator = val_datagen.flow_from_dataframe(
    dataframe=val_gen_df, x_col="filepath", y_col="target_str",
    target_size=(IMG_HEIGHT, IMG_WIDTH), color_mode="grayscale",
    class_mode="binary", classes=["0", "1"], batch_size=BATCH_SIZE,
    shuffle=False
)
assert train_generator.n == len(train_df_resampled), "CONFLIT : images train écartées par Keras"
assert val_generator.n == 3387, "CONFLIT : validation différente du split commun"
assert train_generator.class_indices == val_generator.class_indices == {"0": 0, "1": 1}
print("Batches train:", len(train_generator), "; batches validation:", len(val_generator))


In [ ]:
# CONFLIT — MODIFICATION : 5 epochs max, patience=2, meilleur checkpoint.
# CONFLIT — AJOUT : CSVLogger conserve l'historique après chaque epoch.
checkpoint_path = OUTPUT_DIR / "E3_CNN_best.keras"
callbacks = [
    ModelCheckpoint(str(checkpoint_path), monitor="val_loss", save_best_only=True, verbose=1),
    EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True, verbose=1),
    CSVLogger(str(OUTPUT_DIR / "E3_history_log.csv"))
]
history = model.fit(train_generator, validation_data=val_generator,
                    epochs=5, callbacks=callbacks, verbose=1)
print("Epochs réalisées :", len(history.history["loss"]))
pd.DataFrame(history.history).to_csv(OUTPUT_DIR / "E3_history.csv", index=False)


In [ ]:
# Étape 7 : Évaluation E3 sur validation
# CONFLIT — MODIFICATION : recharger meilleur checkpoint.

In [ ]:
# CONFLIT — MODIFICATION : évaluation uniquement sur validation commune, jamais sur test.
# CONFLIT — AJOUT : rechargement du meilleur checkpoint et reset du générateur.
best_model = tf.keras.models.load_model(str(checkpoint_path))
val_generator.reset()
y_proba = best_model.predict(val_generator, verbose=1).ravel()
y_pred = (y_proba >= 0.5).astype(int)
y_true = np.asarray(val_generator.classes)
assert len(y_true) == len(y_pred) == 3387
print(classification_report(y_true, y_pred, target_names=["Non-COVID", "COVID"], digits=4))
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
ConfusionMatrixDisplay(cm, display_labels=["Non-COVID", "COVID"]).plot(cmap="Blues")
plt.title("E3 — matrice de confusion validation")
plt.savefig(OUTPUT_DIR / "E3_confusion_validation.png", dpi=180, bbox_inches="tight")
plt.show()
pd.DataFrame({"filepath": val_gen_df["filepath"].to_numpy(),
              "y_true": y_true, "y_proba": y_proba, "y_pred": y_pred}).to_csv(
    OUTPUT_DIR / "E3_predictions_validation.csv", index=False)
pd.DataFrame(classification_report(y_true, y_pred,
    target_names=["Non-COVID", "COVID"], output_dict=True)).T.to_csv(
    OUTPUT_DIR / "E3_classification_report_validation.csv")


In [ ]:
# Étape 8 : Courbes E3
# CONFLIT — MODIFICATION : historique réel.

In [ ]:
# CONFLIT — MODIFICATION : historique issu de fit(), jamais recopié à la main.
history_df = pd.DataFrame(history.history)
history_df.index = range(1, len(history_df) + 1)
history_df.index.name = "epoch"
for metric in ["loss", "accuracy", "precision", "recall", "auc"]:
    if metric not in history_df or f"val_{metric}" not in history_df:
        continue
    plt.figure(figsize=(7, 4))
    plt.plot(history_df.index, history_df[metric], marker="o", label="Train")
    plt.plot(history_df.index, history_df[f"val_{metric}"], marker="o", label="Validation")
    plt.title(f"E3 — {metric}")
    plt.xlabel("Epoch")
    plt.legend()
    plt.grid(alpha=.3)
    plt.savefig(OUTPUT_DIR / f"E3_{metric}_curve.png", dpi=180, bbox_inches="tight")
    plt.show()


In [ ]:
# CONFLIT — SUPPRESSION : comparaison interne E1/E3, car E1 est déjà une expérience distincte.
# La comparaison inter-expériences se fera dans le rapport.

In [ ]:
# CONFLIT — SUPPRESSION : courbes comparant E1 et E3 ; E1 n’est pas entraîné ici.